<a href="https://colab.research.google.com/github/Soumo10/signature_detection_-_verification/blob/main/Siamese_Hybrid_Verification_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧬 Signature Verification — Siamese Hybrid Network
### ConvNeXt + Swin Transformer | Contrastive Loss | Google Colab Edition

---

**Goal**: Build an open-set signature verification system that maps the complex structural DNA of a person's handwriting, providing the highest possible defense against counterfeits.

**Architecture**: A Siamese network with twin branches sharing a hybrid backbone (ConvNeXt for local stroke textures + Swin Transformer for global spatial context), trained with contrastive loss.

**Dataset**: 55 writers × 24 genuine + 24 forged signatures = 2640 total images.

### Notebook Outline
| Step | Section | Description |
|------|---------|-------------|
| 1 | Colab Setup & GPU Check | Mount Drive, verify GPU, install deps |
| 2 | Dataset Upload & Exploration | Upload or locate the dataset on Drive |
| 3 | Data Pairing & Loading | Build a Siamese pair dataset split by writer |
| 4 | Model Architecture | Construct the ConvNeXt + Swin hybrid Siamese network |
| 5 | Training | Train with contrastive loss, AdamW, and cosine annealing |
| 6 | Evaluation | Calculate FAR, FRR, EER and plot ROC + distance distributions |
| 7 | Inference Demo | Verify a pair of signatures interactively |
| 8 | Save & Export | Save model to Google Drive |

> ⚠️ **Before running**: Go to **Runtime → Change runtime type → GPU (T4)** to enable GPU acceleration.


---
## Step 1 — Colab Setup & GPU Verification
Mount Google Drive and verify that a GPU is available.

In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Verify GPU
!nvidia-smi

import torch
print(f"\n✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    print("❌ No GPU detected! Go to Runtime → Change runtime type → GPU")

In [ ]:
# Install additional dependencies (PyTorch is pre-installed on Colab)
!pip install timm --quiet

In [ ]:
import os
import random
import numpy as np
from collections import defaultdict
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import timm

from sklearn.metrics import roc_curve, auc
from scipy.optimize import brentq
from scipy.interpolate import interp1d

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ All imports successful. Using: {DEVICE}")

---
## Step 2 — Dataset Upload & Exploration

### Option A: Upload dataset ZIP to Colab (Recommended)
1. Zip the `signatures` folder on your PC (containing `full_org` and `full_forg`).
2. Upload the ZIP to your Google Drive.
3. Set the path below.

### Option B: Upload directly
Use the Colab file uploader (slower for large datasets).

In [ ]:
# ============================================================
#  OPTION A: Dataset is on Google Drive (RECOMMENDED)
#  Upload signatures.zip to your Drive, then set the path:
# ============================================================
DRIVE_ZIP_PATH = "/content/drive/MyDrive/signatures.zip"  # ← EDIT THIS PATH

import zipfile

EXTRACT_DIR = "/content/signatures"

if os.path.exists(DRIVE_ZIP_PATH):
    print(f"📦 Found ZIP at: {DRIVE_ZIP_PATH}")
    print("   Extracting...")
    with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_DIR)
    print(f"   ✅ Extracted to {EXTRACT_DIR}")
else:
    print(f"⚠️  ZIP not found at: {DRIVE_ZIP_PATH}")
    print("   Please upload signatures.zip to your Google Drive and update the path above.")
    print("   Or use Option B below to upload directly.")

In [ ]:
# ============================================================
#  OPTION B: Upload ZIP directly to Colab (skip if using Option A)
# ============================================================
# Uncomment the lines below to upload a ZIP file directly:

# from google.colab import files
# import zipfile
#
# print("Upload your signatures.zip file:")
# uploaded = files.upload()
#
# for filename in uploaded.keys():
#     with zipfile.ZipFile(filename, 'r') as z:
#         z.extractall("/content/signatures")
#     print(f"✅ Extracted {filename}")

In [ ]:
# ============================================================
#  Auto-detect the dataset directories
# ============================================================
def find_dataset(base_path):
    """Search for full_org and full_forg directories."""
    for root, dirs, files in os.walk(base_path):
        if 'full_org' in dirs and 'full_forg' in dirs:
            return root
    return None

BASE_DIR = find_dataset(EXTRACT_DIR)
if BASE_DIR is None:
    # Fallback: try the extract dir itself
    BASE_DIR = EXTRACT_DIR

ORG_DIR  = os.path.join(BASE_DIR, "full_org")
FORG_DIR = os.path.join(BASE_DIR, "full_forg")

assert os.path.isdir(ORG_DIR),  f"❌ full_org not found at {ORG_DIR}"
assert os.path.isdir(FORG_DIR), f"❌ full_forg not found at {FORG_DIR}"

print(f"✅ Dataset found!")
print(f"   Genuine : {ORG_DIR}")
print(f"   Forged  : {FORG_DIR}")

In [ ]:
# Parse images and group by writer ID
def parse_writer_id(filename):
    """Extract writer ID from 'original_10_3.png' or 'forgeries_10_3.png'."""
    parts = filename.split('_')
    if len(parts) >= 3:
        try:
            return int(parts[1])
        except ValueError:
            return -1
    return -1

genuine_by_writer = defaultdict(list)
forged_by_writer  = defaultdict(list)

for f in sorted(os.listdir(ORG_DIR)):
    if f.lower().endswith('.png'):
        wid = parse_writer_id(f)
        if wid != -1:
            genuine_by_writer[wid].append(os.path.join(ORG_DIR, f))

for f in sorted(os.listdir(FORG_DIR)):
    if f.lower().endswith('.png'):
        wid = parse_writer_id(f)
        if wid != -1:
            forged_by_writer[wid].append(os.path.join(FORG_DIR, f))

all_writers = sorted(genuine_by_writer.keys())
total_genuine = sum(len(v) for v in genuine_by_writer.values())
total_forged  = sum(len(v) for v in forged_by_writer.values())

print(f"📊 Dataset Summary")
print(f"{'='*40}")
print(f"  Total writers       : {len(all_writers)}")
print(f"  Genuine signatures  : {total_genuine}")
print(f"  Forged signatures   : {total_forged}")
print(f"  Total images        : {total_genuine + total_forged}")
print(f"  Samples per writer  : {len(genuine_by_writer[all_writers[0]])} genuine, {len(forged_by_writer[all_writers[0]])} forged")

In [ ]:
# Visualize samples from 3 writers
sample_writers = all_writers[:3]
fig, axes = plt.subplots(3, 4, figsize=(20, 12))
fig.suptitle("Sample Signatures (Genuine vs Forged)", fontsize=18, fontweight='bold', y=1.02)

for row, wid in enumerate(sample_writers):
    for col in range(2):
        img = mpimg.imread(genuine_by_writer[wid][col])
        axes[row][col].imshow(img, cmap='gray')
        axes[row][col].set_title(f"Writer {wid} — Genuine", color='green', fontweight='bold')
        axes[row][col].axis('off')
    for col in range(2, 4):
        img = mpimg.imread(forged_by_writer[wid][col - 2])
        axes[row][col].imshow(img, cmap='gray')
        axes[row][col].set_title(f"Writer {wid} — Forged", color='red', fontweight='bold')
        axes[row][col].axis('off')

plt.tight_layout()
plt.show()

---
## Step 3 — Data Pairing & Loading

### Writer-based Split (Open-Set Verification)
- **Train**: Writers 1–40 (40 writers)
- **Validation**: Writers 41–45 (5 writers)
- **Test**: Writers 46–55 (10 writers)

### Pair Types
- **Positive (Label 0)**: Genuine + Genuine from same writer → distance should be small.
- **Negative (Label 1)**: Genuine + Forged for same writer → distance should be large.

In [ ]:
class SignaturePairDataset(Dataset):
    """
    Generates balanced signature pairs for Siamese training.
    Label 0 = Genuine-Genuine pair (should be CLOSE)
    Label 1 = Genuine-Forged pair  (should be FAR)
    """
    def __init__(self, writer_ids, genuine_dict, forged_dict,
                 pairs_per_epoch=2000, transform=None):
        self.writer_ids = writer_ids
        self.genuine_dict = genuine_dict
        self.forged_dict = forged_dict
        self.pairs_per_epoch = pairs_per_epoch
        self.transform = transform
        self._generate_pairs()

    def _generate_pairs(self):
        self.pairs = []
        half = self.pairs_per_epoch // 2

        for _ in range(half):
            wid = random.choice(self.writer_ids)
            a, b = random.sample(self.genuine_dict[wid], 2)
            self.pairs.append((a, b, 0))

        for _ in range(half):
            wid = random.choice(self.writer_ids)
            a = random.choice(self.genuine_dict[wid])
            b = random.choice(self.forged_dict[wid])
            self.pairs.append((a, b, 1))

        random.shuffle(self.pairs)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        path1, path2, label = self.pairs[idx]
        img1 = Image.open(path1).convert('RGB')
        img2 = Image.open(path2).convert('RGB')

        if self.transform:
            img1 = self.transform(img1)
            img2 = self.transform(img2)

        return img1, img2, torch.tensor([label], dtype=torch.float32)

In [ ]:
# Image Transforms
IMG_SIZE = 224

transform_train = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, scale=(0.9, 1.1)),
    transforms.ColorJitter(contrast=0.2, brightness=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

transform_eval = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Writer Split
train_writers = [w for w in all_writers if w <= 40]
val_writers   = [w for w in all_writers if 41 <= w <= 45]
test_writers  = [w for w in all_writers if w >= 46]

print(f"Writer Split:")
print(f"  Train : {len(train_writers)} writers  (IDs: {train_writers[0]}–{train_writers[-1]})")
print(f"  Val   : {len(val_writers)} writers   (IDs: {val_writers[0]}–{val_writers[-1]})")
print(f"  Test  : {len(test_writers)} writers  (IDs: {test_writers[0]}–{test_writers[-1]})")

# Create Datasets & Loaders
BATCH_SIZE = 16

train_dataset = SignaturePairDataset(
    train_writers, genuine_by_writer, forged_by_writer,
    pairs_per_epoch=4000, transform=transform_train
)
val_dataset = SignaturePairDataset(
    val_writers, genuine_by_writer, forged_by_writer,
    pairs_per_epoch=500, transform=transform_eval
)
test_dataset = SignaturePairDataset(
    test_writers, genuine_by_writer, forged_by_writer,
    pairs_per_epoch=1000, transform=transform_eval
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"\n📦 DataLoaders Created:")
print(f"  Train : {len(train_dataset)} pairs → {len(train_loader)} batches")
print(f"  Val   : {len(val_dataset)} pairs → {len(val_loader)} batches")
print(f"  Test  : {len(test_dataset)} pairs → {len(test_loader)} batches")

---
## Step 4 — Model Architecture

1. **ConvNeXt Tiny** — Local stroke textures, pen pressure, fine details.
2. **Swin Transformer Tiny** — Global spatial relationships, handwriting flow.
3. **Fusion MLP** — Concatenates both → 512-D embedding.
4. **L2 Normalization** — Stable distance computation.

In [ ]:
class HybridSiameseNetwork(nn.Module):
    """
    Siamese Network with ConvNeXt + Swin Transformer hybrid backbone.
    Both branches share identical weights.
    """
    def __init__(self, embedding_dim=512):
        super().__init__()

        # Backbone 1: ConvNeXt (local CNN features)
        self.convnext = timm.create_model(
            'convnext_tiny', pretrained=True, num_classes=0
        )
        convnext_dim = self.convnext.num_features

        # Backbone 2: Swin Transformer (global attention features)
        self.swin = timm.create_model(
            'swin_tiny_patch4_window7_224', pretrained=True, num_classes=0
        )
        swin_dim = self.swin.num_features

        total_features = convnext_dim + swin_dim

        # Fusion MLP Head
        self.fusion = nn.Sequential(
            nn.Linear(total_features, 1024),
            nn.BatchNorm1d(1024),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(1024, embedding_dim),
        )

        print(f"✅ HybridSiameseNetwork initialized")
        print(f"   ConvNeXt dim  : {convnext_dim}")
        print(f"   Swin dim      : {swin_dim}")
        print(f"   Fused dim     : {total_features}")
        print(f"   Embedding dim : {embedding_dim}")

    def forward_once(self, x):
        feat_cnn = self.convnext(x)
        feat_swin = self.swin(x)
        fused = torch.cat([feat_cnn, feat_swin], dim=1)
        embedding = self.fusion(fused)
        embedding = F.normalize(embedding, p=2, dim=1)
        return embedding

    def forward(self, img1, img2):
        return self.forward_once(img1), self.forward_once(img2)

In [ ]:
model = HybridSiameseNetwork(embedding_dim=512).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n📐 Model Parameters:")
print(f"   Total      : {total_params:,}")
print(f"   Trainable  : {trainable_params:,}")

---
## Step 5 — Training

**Contrastive Loss**:  
$\mathcal{L} = (1 - y) \cdot D^2 + y \cdot \max(0, m - D)^2$

- $y = 0$: genuine pair → minimize distance
- $y = 1$: forged pair → push distance beyond margin $m$

In [ ]:
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=2.0):
        super().__init__()
        self.margin = margin

    def forward(self, emb1, emb2, label):
        distance = F.pairwise_distance(emb1, emb2, keepdim=True)
        loss = torch.mean(
            (1 - label) * torch.pow(distance, 2) +
            label * torch.pow(torch.clamp(self.margin - distance, min=0.0), 2)
        )
        return loss

In [ ]:
NUM_EPOCHS   = 15
LR           = 1e-4
WEIGHT_DECAY = 1e-4
MARGIN       = 2.0

criterion = ContrastiveLoss(margin=MARGIN)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

print(f"⚙️  Training Config:")
print(f"   Epochs       : {NUM_EPOCHS}")
print(f"   Learning Rate: {LR}")
print(f"   Optimizer    : AdamW")
print(f"   Scheduler    : CosineAnnealingLR")
print(f"   Loss Margin  : {MARGIN}")
print(f"   Batch Size   : {BATCH_SIZE}")
print(f"   Device       : {DEVICE}")

In [ ]:
# ============================================================
# TRAINING LOOP
# ============================================================
train_losses = []
val_losses   = []
best_val_loss = float('inf')

# Save best model to Google Drive so it persists
SAVE_DIR = "/content/drive/MyDrive/signature_model"
os.makedirs(SAVE_DIR, exist_ok=True)
SAVE_PATH = os.path.join(SAVE_DIR, "siamese_hybrid_best.pth")

print(f"\n🚀 Starting Training...\n")
print(f"{'Epoch':>6} | {'Train Loss':>12} | {'Val Loss':>12} | {'LR':>12} | {'Status'}")
print(f"{'-'*70}")

for epoch in range(NUM_EPOCHS):
    # ---- Train ----
    model.train()
    epoch_train_loss = 0.0

    for img1, img2, label in train_loader:
        img1  = img1.to(DEVICE)
        img2  = img2.to(DEVICE)
        label = label.to(DEVICE)

        optimizer.zero_grad()
        emb1, emb2 = model(img1, img2)
        loss = criterion(emb1, emb2, label)
        loss.backward()
        optimizer.step()

        epoch_train_loss += loss.item()

    avg_train_loss = epoch_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # ---- Validate ----
    model.eval()
    epoch_val_loss = 0.0

    with torch.no_grad():
        for img1, img2, label in val_loader:
            img1  = img1.to(DEVICE)
            img2  = img2.to(DEVICE)
            label = label.to(DEVICE)
            emb1, emb2 = model(img1, img2)
            loss = criterion(emb1, emb2, label)
            epoch_val_loss += loss.item()

    avg_val_loss = epoch_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)

    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step()

    # Save best
    status = ""
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), SAVE_PATH)
        status = "✅ Best saved!"

    print(f"{epoch+1:>6} | {avg_train_loss:>12.4f} | {avg_val_loss:>12.4f} | {current_lr:>12.6f} | {status}")

print(f"\n🎉 Training complete! Best val loss: {best_val_loss:.4f}")
print(f"   Model saved to Google Drive: {SAVE_PATH}")

In [ ]:
# Plot Training Curves
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, NUM_EPOCHS + 1), train_losses, 'o-', label='Train Loss', color='#1f77b4')
ax.plot(range(1, NUM_EPOCHS + 1), val_losses,   's-', label='Val Loss',   color='#ff7f0e')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Contrastive Loss', fontsize=12)
ax.set_title('Training & Validation Loss', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Step 6 — Evaluation

Evaluate on **unseen test writers** (46–55) using FAR, FRR, EER, and ROC.

In [ ]:
# Load best model
model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
model.eval()
print(f"✅ Loaded best model from: {SAVE_PATH}")

# Compute distances on test set
all_distances = []
all_labels    = []

with torch.no_grad():
    for img1, img2, label in test_loader:
        img1 = img1.to(DEVICE)
        img2 = img2.to(DEVICE)
        emb1, emb2 = model(img1, img2)
        dist = F.pairwise_distance(emb1, emb2)
        all_distances.extend(dist.cpu().numpy())
        all_labels.extend(label.cpu().numpy().flatten())

all_distances = np.array(all_distances)
all_labels    = np.array(all_labels)

print(f"\n📊 Computed distances for {len(all_distances)} test pairs.")
print(f"   Genuine pairs : {int(np.sum(all_labels == 0))}")
print(f"   Forged pairs  : {int(np.sum(all_labels == 1))}")

In [ ]:
# Calculate EER and optimal threshold
fpr, tpr, thresholds = roc_curve(all_labels, all_distances, pos_label=1)
roc_auc = auc(fpr, tpr)

eer = brentq(lambda x: 1.0 - x - interp1d(fpr, tpr)(x), 0.0, 1.0)
eer_threshold = float(interp1d(fpr, thresholds)(eer))

print(f"\n🎯 Evaluation Results (Test Set — Unseen Writers 46–55):")
print(f"{'='*50}")
print(f"  Equal Error Rate (EER)     : {eer:.2%}")
print(f"  Optimal Distance Threshold : {eer_threshold:.4f}")
print(f"  ROC AUC                    : {roc_auc:.4f}")
print(f"{'='*50}")
print(f"\n  → distance < {eer_threshold:.4f} : SAME person (Genuine)")
print(f"  → distance ≥ {eer_threshold:.4f} : DIFFERENT (Potential Forgery)")

In [ ]:
# Plot ROC Curve + Distance Distributions
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# ROC Curve
axes[0].plot(fpr, tpr, color='#2196F3', lw=2, label=f'ROC Curve (AUC = {roc_auc:.3f})')
axes[0].plot([0, 1], [0, 1], '--', color='gray', lw=1, label='Random Baseline')
axes[0].plot(eer, 1 - eer, 'ro', markersize=10, label=f'EER = {eer:.2%}')
axes[0].set_xlabel('False Positive Rate (FAR)', fontsize=12)
axes[0].set_ylabel('True Positive Rate (1 - FRR)', fontsize=12)
axes[0].set_title('ROC Curve', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Distance Distributions
genuine_dists = all_distances[all_labels == 0]
forged_dists  = all_distances[all_labels == 1]

sns.kdeplot(genuine_dists, fill=True, color='green', alpha=0.5, label='Genuine Pairs', ax=axes[1])
sns.kdeplot(forged_dists,  fill=True, color='red',   alpha=0.5, label='Forged Pairs',  ax=axes[1])
axes[1].axvline(x=eer_threshold, color='black', linestyle='--', lw=2, label=f'Threshold ({eer_threshold:.2f})')
axes[1].set_xlabel('Euclidean Distance', fontsize=12)
axes[1].set_ylabel('Density', fontsize=12)
axes[1].set_title('Distance Distribution: Genuine vs Forged', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# FAR and FRR at various thresholds
print(f"\n📈 FAR / FRR at different thresholds:")
print(f"{'Threshold':>12} | {'FAR':>8} | {'FRR':>8} | {'Note'}")
print(f"{'-'*50}")

for t in np.linspace(all_distances.min(), all_distances.max(), 10):
    far = np.mean(forged_dists < t)
    frr = np.mean(genuine_dists >= t)
    marker = " ← EER" if abs(t - eer_threshold) < 0.05 else ""
    print(f"{t:>12.4f} | {far:>7.2%} | {frr:>7.2%} | {marker}")

---
## Step 7 — Inference Demo
Test on specific signature pairs.

In [ ]:
def verify_signature_pair(model, img_path1, img_path2, threshold, transform, device):
    model.eval()
    img1 = Image.open(img_path1).convert('RGB')
    img2 = Image.open(img_path2).convert('RGB')
    img1_t = transform(img1).unsqueeze(0).to(device)
    img2_t = transform(img2).unsqueeze(0).to(device)

    with torch.no_grad():
        emb1, emb2 = model(img1_t, img2_t)
        distance = F.pairwise_distance(emb1, emb2).item()
    return distance, distance < threshold


# Demo with test writer
demo_writer = test_writers[0]
gen1  = genuine_by_writer[demo_writer][0]
gen2  = genuine_by_writer[demo_writer][1]
forg1 = forged_by_writer[demo_writer][0]

dist_gg, same_gg = verify_signature_pair(model, gen1, gen2,  eer_threshold, transform_eval, DEVICE)
dist_gf, same_gf = verify_signature_pair(model, gen1, forg1, eer_threshold, transform_eval, DEVICE)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle(f"Verification Demo — Writer {demo_writer} (Threshold: {eer_threshold:.3f})",
             fontsize=16, fontweight='bold')

axes[0].imshow(mpimg.imread(gen1), cmap='gray')
axes[0].set_title('Reference\n(Genuine)', fontweight='bold')
axes[0].axis('off')

color = 'green' if same_gg else 'red'
verdict = 'SAME ✅' if same_gg else 'DIFFERENT ❌'
axes[1].imshow(mpimg.imread(gen2), cmap='gray')
axes[1].set_title(f'Query (Genuine)\nDist: {dist_gg:.3f} → {verdict}', color=color, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(mpimg.imread(gen1), cmap='gray')
axes[2].set_title('Reference\n(Genuine)', fontweight='bold')
axes[2].axis('off')

color = 'green' if same_gf else 'red'
verdict = 'SAME ✅' if same_gf else 'DIFFERENT ❌'
axes[3].imshow(mpimg.imread(forg1), cmap='gray')
axes[3].set_title(f'Query (Forged)\nDist: {dist_gf:.3f} → {verdict}', color=color, fontweight='bold')
axes[3].axis('off')

plt.tight_layout()
plt.show()

---
## Step 8 — Save & Export
Model is saved to Google Drive. You can also download it locally.

In [ ]:
print(f"📁 Model saved on Google Drive:")
print(f"   {SAVE_PATH}")
print(f"   Size: {os.path.getsize(SAVE_PATH) / (1024*1024):.1f} MB")

# Optional: Download the model to your local machine
# from google.colab import files
# files.download(SAVE_PATH)

print("\n🎉 All done! Your Siamese Hybrid Signature Verification model is trained and saved.")